# Chapitre 4 — EDA diagnostique et qualité des données

**Durée estimée : 8-10 heures**

---

## Objectifs d'apprentissage

À la fin de ce chapitre, vous serez capable de :

1. **Distinguer** l'EDA diagnostique (trouver les problèmes) de l'EDA analytique (comprendre les patterns)
2. **Évaluer** la qualité des données selon les 6 dimensions standardisées
3. **Détecter** les valeurs manquantes, doublons et outliers avec les outils pandas appropriés
4. **Documenter** les problèmes identifiés dans une checklist de qualité priorisée

---

## 4.5 Détection des doublons

### Types de doublons

| Type | Définition | Exemple |
|------|------------|--------|
| **Exact** | Lignes 100% identiques | Import dupliqué |
| **Quasi-doublon** | Même entité, petites différences | "Jean Dupont" vs "Jean DUPONT" |
| **Doublon partiel** | Même clé, valeurs différentes | Même client avec 2 adresses |

### Détecter les doublons exacts

In [ ]:
import pandas as pd
import numpy as np

In [23]:
# Créer un DataFrame avec des doublons
df_doublons = pd.DataFrame({
    'id': [1, 2, 3, 4, 5, 1, 2],
    'nom': ['Alice', 'Bob', 'Charlie', 'David', 'Eve', 'Alice', 'Bob'],
    'email': ['a@t.com', 'b@t.com', 'c@t.com', 'd@t.com', 'e@t.com', 'a@t.com', 'b@t.com'],
    'age': [25, 30, 35, 40, 28, 25, 30]
})

print("DataFrame avec doublons :")
df_doublons

DataFrame avec doublons :


,id,nom,email,age
0,1,Alice,a@t.com,25
1,2,Bob,b@t.com,30
2,3,Charlie,c@t.com,35
3,4,David,d@t.com,40
4,5,Eve,e@t.com,28
5,1,Alice,a@t.com,25
6,2,Bob,b@t.com,30


In [24]:
# Nombre de doublons
print(f"Doublons exacts : {df_doublons.duplicated().sum()}")

Doublons exacts : 2


In [ ]:
# Voir les doublons (keep=False montre tous les doublons)
doublons = df_doublons[df_doublons.duplicated(keep=False)]
# keep=False montre toutes les occurrences des lignes dupliquées
print("Lignes dupliquées :")
doublons.sort_values(by=['id'])

Lignes dupliquées :


,id,nom,email,age
0,1,Alice,a@t.com,25
5,1,Alice,a@t.com,25
1,2,Bob,b@t.com,30
6,2,Bob,b@t.com,30


In [26]:
# Doublons sur une clé spécifique
doublons_id = df_doublons[df_doublons.duplicated(subset=['id'], keep=False)]
print(f"\nDoublons sur 'id' : {df_doublons.duplicated(subset=['id']).sum()}")
print(doublons_id)


Doublons sur 'id' : 2
   id    nom    email  age
0   1  Alice  a@t.com   25
1   2    Bob  b@t.com   30
5   1  Alice  a@t.com   25
6   2    Bob  b@t.com   30


In [27]:
# Doublons sur email
doublons_email = df_doublons[df_doublons.duplicated(subset=['email'], keep=False)]
print(f"Emails dupliqués : {doublons_email['email'].nunique()}")

Emails dupliqués : 2


### Identifier les clés candidates

In [28]:
def verifier_cle_unique(df, colonnes):
    """Vérifie si un ensemble de colonnes peut servir de clé unique."""
    doublons = df.duplicated(subset=colonnes)
    if doublons.sum() == 0:
        print(f"✅ {colonnes} peut servir de clé unique")
    else:
        print(f"❌ {colonnes} a {doublons.sum()} doublons")

# Tests
verifier_cle_unique(df_doublons, ['id'])
verifier_cle_unique(df_doublons, ['email'])
verifier_cle_unique(df_doublons, ['nom', 'age'])

❌ ['id'] a 2 doublons
❌ ['email'] a 2 doublons
❌ ['nom', 'age'] a 2 doublons


### ✍️ Exercice 4.5 : Chasse aux doublons (10 min)

In [29]:
import pandas as pd

df_chasse = pd.DataFrame({
    'client_id': [1, 2, 3, 4, 5, 1],
    'nom': ['Alice Martin', 'Bob Dupont', 'Charlie Brown', 'David Lee', 'Eve Wilson', 'Alice MARTIN'],
    'email': ['alice@test.com', 'bob@test.com', 'charlie@test.com', 'david@test.com', 'eve@test.com', 'alice@test.com'],
    'date_achat': ['2024-01-15', '2024-01-16', '2024-01-17', '2024-01-18', '2024-01-19', '2024-01-20'],
    'montant': [100, 200, 150, 300, 250, 100]
})

print("DataFrame :")
df_chasse

DataFrame :


,client_id,nom,email,date_achat,montant
0,1,Alice Martin,alice@test.com,2024-01-15,100
1,2,Bob Dupont,bob@test.com,2024-01-16,200
2,3,Charlie Brown,charlie@test.com,2024-01-17,150
3,4,David Lee,david@test.com,2024-01-18,300
4,5,Eve Wilson,eve@test.com,2024-01-19,250
5,1,Alice MARTIN,alice@test.com,2024-01-20,100


In [30]:
# Votre analyse
# 1. Combien de doublons exacts ?
print("Doublons exacts :", df_chasse.duplicated().sum())

# 2. Combien de doublons sur client_id ?
print("Doublons client_id :", df_chasse.duplicated(subset=['client_id']).sum())

# 3. Combien de doublons sur email ?
print("Doublons email :", df_chasse.duplicated(subset=['email']).sum())

# 4. Y a-t-il un quasi-doublon sur le nom ? Comment le détecteriez-vous ?
print("\nDétection quasi-doublons sur nom (normalisation) :")
df_chasse['nom_normalise'] = df_chasse['nom'].str.lower().str.strip()
print("Doublons après normalisation :", df_chasse.duplicated(subset=['nom_normalise']).sum())

Doublons exacts : 0
Doublons client_id : 1
Doublons email : 1

Détection quasi-doublons sur nom (normalisation) :
Doublons après normalisation : 1


> 💭 **Question Socratique #3** : Si deux lignes ont le même email mais des noms légèrement différents ("Jean Dupont" vs "jean dupont"), est-ce un doublon à supprimer ou deux entrées légitimes ? Comment décideriez-vous ?

> 💭 **Réponse* :

1️⃣ Les points à considérer pour décider

a) Identifiant unique fiable

	•	L’email peut être un identifiant unique si l’entreprise utilise les emails pour login ou contact.
	•	Si l’email est unique par définition métier → probable doublon, les variations de nom peuvent être dues à :
	•	majuscules/minuscules
	•	erreurs de saisie

b) Variation réelle du nom

	•	Vérifier si la différence est mineure (casse, accents, espaces) ou substantielle (orthographe différente).
	•	Dans ton exemple, c’est juste la casse → souvent considéré comme identique.

c) Contexte métier

	•	Certains systèmes autorisent plusieurs personnes à partager un email (famille, service client général).
	•	Il faut comprendre la règle métier spécifique :
	•	Email = identifiant unique du client ?
	•	Ou email = contact partagé entre plusieurs clients ?


2️⃣ Stratégie de décision

	1.	Normaliser les données :
	•	Minuscule, retirer accents et espaces → "jean dupont"
	2.	Comparer les autres champs :
	•	Adresse, téléphone, date de naissance… si tout concorde → fusionner / supprimer doublon
	•	Si divergence importante → conserver comme entrées distinctes
	3.	Documentation et règles métier :
	•	Mettre en place une règle : "email unique = fusionner" ou "email partagé = conserver"


3️⃣ Conclusion pédagogique

Ce n’est pas automatique :
	•	Les doublons doivent être jugés selon l’identifiant unique et le contexte métier
	•	Une décision robuste implique validation par les règles métier et parfois revérification manuelle

Schéma décisionnel pour identifier un doublon

1️⃣ Deux lignes ont le même email ?

    ├─ Non → Ce n’est pas un doublon → Conserver les lignes
    └─ Oui → Passer à l’étape suivante

2️⃣ Normaliser les données (casse, accents, espaces) pour le nom

    ├─ Les noms deviennent identiques → Passer à l’étape suivante
    └─ Les noms restent différents → Vérifier les autres champs (téléphone, adresse, date de naissance)

3️⃣ Vérifier la concordance des autres champs

    ├─ Champs concordants → Probable doublon → Fusionner ou supprimer selon règle métier
    └─ Champs divergents → Lignes distinctes → Conserver les deux

4️⃣ Contexte métier spécifique

    ├─ Email = identifiant unique par client → Fusionner doublon
    └─ Email partagé (famille, entreprise) → Conserver plusieurs lignes